# PyVista trame through `jupyter-loopback`

This notebook shows `jupyter-loopback` standing in for `jupyter-server-proxy` when wiring PyVista's `trame` Jupyter backend. The trame server runs in this kernel on a `127.0.0.1:<port>` loopback socket; the iframe in this notebook needs the browser to reach that socket. With `jupyter-loopback`, the user does nothing.

Three moving parts:

1. The `pyvista_loopback_demo._jupyter` server extension is auto-loaded by JupyterLab. It calls `setup_proxy_handler(web_app, namespace='pyvista')` so the route `<base_url>/pyvista-proxy/<port>/...` proxies HTTP and WebSocket to `127.0.0.1:<port>` in the kernel.
2. `pyvista_loopback_demo.configure()` reads `autodetect_prefix('pyvista')` and sets `pv.global_theme.trame.server_proxy_prefix` so PyVista's `build_url` produces iframe URLs like `<base>/pyvista-proxy/<port>/index.html`.
3. The trame frontend (vue3-www `index.html`) opens its WS at a path relative to `window.location.pathname`, which lands at `<base>/pyvista-proxy/<port>/ws`. The same proxy route handles it: upgrade to WS, forward to the kernel-side trame server.

If the plot below is interactive (you can rotate it with the mouse), the round-trip works end to end.

End-to-end automated validation (HTTP and WebSocket) lives in `demos/test_pyvista_proxy.py`. It's a single-process smoke test that doesn't need a browser.

## 1. The server extension

`pyvista_loopback_demo._jupyter` is one line that mounts the proxy. Auto-enabled via `jupyter-config/jupyter_server_config.d/pyvista-loopback-demo.json`.

In [1]:
import inspect

from pyvista_loopback_demo import _jupyter

print(inspect.getsource(_jupyter))

"""
Jupyter-server extension that mounts the jupyter-loopback proxy for the
``pyvista`` namespace, so the trame server PyVista launches in the
kernel becomes reachable from the notebook browser.
"""

from jupyter_loopback import setup_proxy_handler


def _jupyter_server_extension_points():
    return [{"module": "pyvista_loopback_demo._jupyter"}]


def _load_jupyter_server_extension(server_app):
    setup_proxy_handler(server_app.web_app, namespace="pyvista")
    server_app.log.info(
        "pyvista-loopback-demo: proxy mounted at %spyvista-proxy/<port>/...",
        server_app.web_app.settings.get("base_url", "/"),
    )



Confirm the proxy is mounted by hitting the `__probe__` route the extension installs at `<base>/pyvista-proxy/__probe__`. The probe handler is mounted on jupyter-server (not the kernel), so it answers without forwarding. That makes it safe to call from the kernel without deadlocking against the in-kernel trame server.

A `204` means the extension loaded; a `404` means it didn't (the comm bridge would take over in that case).

In [2]:
from jupyter_server.serverapp import list_running_servers
import requests

servers = list(list_running_servers())
if not servers:
    raise RuntimeError("no running jupyter-server visible from this kernel")
server = servers[0]
base_url = server["url"].rstrip("/")
token = server.get("token", "")
probe_url = f"{base_url}/pyvista-proxy/__probe__"
r = requests.head(probe_url, params={"token": token} if token else None, timeout=5)
print("probe status:", r.status_code)
print("namespace header:", r.headers.get("X-Jupyter-Loopback-Namespace"))
assert r.status_code == 204, f"probe failed: {r.status_code}"

RuntimeError: no running jupyter-server visible from this kernel

## 2. Wire PyVista to the proxy

`autodetect_prefix('pyvista')` returns a root-absolute template like `/pyvista-proxy/{port}` (or, on JupyterHub, `/user/<name>/pyvista-proxy/{port}`). PyVista's `build_url` concatenates the prefix directly with the integer port, so we strip the `{port}` placeholder and keep the trailing slash.

In [ ]:
import pyvista as pv
from pyvista_loopback_demo import configure, loopback_prefix_for_pyvista

from jupyter_loopback import autodetect_prefix

print("autodetect_prefix('pyvista') ->", autodetect_prefix("pyvista"))
print("prefix passed to PyVista     ->", loopback_prefix_for_pyvista())

applied = configure()
print("server_proxy_enabled         ->", pv.global_theme.trame.server_proxy_enabled)
print("server_proxy_prefix          ->", pv.global_theme.trame.server_proxy_prefix)
assert applied is not None, "configure() returned None; not running in a Jupyter kernel?"

## 3. Render a plot

`set_jupyter_backend('trame')` launches a trame server on `127.0.0.1:<port>` and `Plotter.show` returns an iframe widget pointing at the proxy URL. PyVista is in off-screen mode (set by `PYVISTA_OFF_SCREEN=true` in the Docker image) so VTK uses the system's GL libraries without an X server.

If the iframe URL contains `pyvista-proxy/<port>` and the plot is interactive in the browser, the proxy is doing its job. The browser opens the iframe at that URL, and from inside the iframe the trame frontend opens a WebSocket back through the same proxy URL. Both hops go through `LoopbackProxyHandler`.

In [ ]:
pv.set_jupyter_backend("trame")

mesh = pv.Wavelet()
pl = pv.Plotter(notebook=True)
pl.add_mesh(mesh, show_edges=True, scalars="RTData")
pl.add_axes()
widget = pl.show(jupyter_backend="trame", return_viewer=True)
print("iframe src:", widget.src)
assert "/pyvista-proxy/" in widget.src, widget.src
widget

## What was validated

- `pyvista_loopback_demo._jupyter` extension auto-loaded; the probe at `<base>/pyvista-proxy/__probe__` answers `204`.
- `autodetect_prefix('pyvista')` returns the right root-absolute template; `configure()` produces the trailing-slash prefix that PyVista's `build_url` expects.
- `Plotter.show` produces an iframe whose `src` is rooted at `/pyvista-proxy/<port>/...` with no `127.0.0.1`.

What this notebook does *not* validate from the kernel: the actual HTTP and WebSocket round-trip through the proxy. From inside the same kernel that hosts trame, a sync request would block trame's coroutines on the same asyncio loop and deadlock. That round-trip is exercised in `demos/test_pyvista_proxy.py`, which uses async I/O end to end and asserts both:

- `GET <proxy>/pyvista-proxy/<port>/index.html` returns the trame index HTML (status `200`).
- `WS  <proxy>/pyvista-proxy/<port>/ws` accepts a real wslink `system:hello` and replies with a `clientID`.

Plus this notebook's iframe `src` points at `/pyvista-proxy/...`, so the browser path that runs inside the iframe is exercising the same code.